# ZeroPoint against hosted LLMs

Runs Anthropic and OpenAI models as decision engines on the same eight tests as `benchmarks/compare`, and scores them
next to ZeroPoint, Laya, Semantic Router and zero-shot NLI with the same `report.py`.

How each model is asked is fixed in `benchmarks/llm/run_llm.py`: the text and the options plus *none of these*, one
forced tool call (Anthropic) or strict JSON schema (OpenAI) with the choice as an enum and a stated confidence from 0
to 1. No examples, no reasoning step. Each test reads the same fixed 300-case sample as zero-shot NLI, then the same
cases again with the options in another order.

You need API keys. Each model makes about 4,800 calls (8 tests × 300 cases × 2 orders).

In [ ]:
!git clone -q https://github.com/Javihaus/zeropoint.git
%cd zeropoint
!pip install -q -e . anthropic openai datasets pandas scikit-learn matplotlib

## 1. Rebuild the tests

The test texts are not in the repository; this rebuilds them from pinned dataset versions and checks their hashes.

In [ ]:
!python benchmarks/compare/build_tests.py
!cd benchmarks/compare/tests && sha256sum -c SHA256SUMS

## 2. Keys and prices

Prices in USD per million tokens (input, output), from each provider's price page on the day you run this. They
are used only to compute the cost per decision.

In [ ]:
import os, getpass
if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (leave empty to skip): ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (leave empty to skip): ")

ANTHROPIC_MODELS = ["claude-opus-5-5", "claude-sonnet-5-5", "claude-haiku-5-5"]
OPENAI_MODELS = []          # the exact model ids, e.g. the Luna, Terra and Sol models from OpenAI's model list

PRICES = {                  # (input, output) USD per million tokens: fill in before running
    "claude-opus-5-5": (None, None),
    "claude-sonnet-5-5": (None, None),
    "claude-haiku-5-5": (None, None),
}

In [ ]:
import re
src = open("benchmarks/llm/run_llm.py").read()
src = re.sub(r"PRICES = \{.*?\n\}", "PRICES = " + repr(PRICES).replace("), ", "),\n    "), src, flags=re.S)
open("benchmarks/llm/run_llm.py", "w").write(src)

## 3. Run the models

Results are written as they come; if a run stops, run the cell again and it continues.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    for m in ANTHROPIC_MODELS:
        !python benchmarks/llm/run_llm.py anthropic {m}
if os.environ.get("OPENAI_API_KEY"):
    for m in OPENAI_MODELS:
        !python benchmarks/llm/run_llm.py openai {m}

## 4. Score everything together

In [ ]:
!python benchmarks/compare/report.py > /dev/null
from IPython.display import Markdown
Markdown(open("benchmarks/compare/results/tables.md").read())

## 5. Accuracy against cost and latency

On the 300-case sample of every test, so every point comes from the same cases. ZeroPoint, Laya, Semantic Router and
zero-shot NLI run locally on a CPU: their cost per decision is the price of CPU time, close to zero, and is drawn at
the left edge.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
S = json.load(open("benchmarks/compare/results/summary.json"))["systems"]
def mean(r, k):
    v = [t["on_sample"].get(k) for t in r["tests"].values()]
    v = [x for x in v if x is not None]
    return float(np.mean(v)) if v else None
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for s, r in S.items():
    acc, silent = mean(r, "decision"), mean(r, "sure_wrong")
    cost = r.get("cost_usd_per_decision") or 1e-7
    for a, x in [(ax[0], cost), (ax[1], r["ms_median_all"])]:
        a.scatter(x, 100 * acc, s=60)
        a.annotate(f"{r['name']}" + (f"\nsilent mistakes {100 * silent:.0f}%" if silent is not None else ""),
                   (x, 100 * acc), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax[0].set(xscale="log", xlabel="cost per decision, USD (log; local models at the left edge)",
          ylabel="right decision, none included (%)")
ax[1].set(xscale="log", xlabel="median latency per decision, ms (log)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("benchmarks/llm/accuracy_cost_latency.png", dpi=150)